# PDD vs 크리깅 — 꺾인(비매끄러운) 응답면 비교

지도교수 지침(GUIDELINE.md 참고): 범퍼 충돌 응답은 좌굴모드 전환 등으로 설계변수에 대해 매끄럽지 않게(꺾이며) 변할 수 있어서, 전역 다항식인 PDD보다 국소 적응형인 크리깅이 서로게이트로 더 적합할 수 있음. 이 노트북은 그 주장을 실제 카티아/아바쿠스 데이터 없이도 **합성 벤치마크 함수로 미리 검증**해보는 용도.

방법: 일부러 꺾임(kink)이 있는 3변수 함수를 만들고, 같은 학습 표본으로 PDD와 크리깅을 각각 적합 → 같은 검증 표본으로 예측 정확도(R², RMSE) 비교.

주의: 실제 SEA가 정확히 이 수식을 따른다는 뜻은 아니고, "좌굴모드 전환처럼 매끄럽지 않은 응답"의 성질만 흉내낸 것.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

rng_seed = 0

# ---- 핵심 PDD 함수 (PDD_Legendre_ver4.ipynb에서 그대로 가져옴) ----

def theoretical_scale(x, domain_min, domain_max, target_min=-1, target_max=1):
    x_std = (x - domain_min) / (domain_max - domain_min)
    return x_std * (target_max - target_min) + target_min


def basis(x, a):
    if a == 0:
        return np.ones_like(x)
    if a == 1:
        return x
    p_prev2 = np.ones_like(x)
    p_prev1 = x.copy()
    p_n = None
    for n in range(1, a):
        p_n = ((2 * n + 1) * x * p_prev1 - n * p_prev2) / (n + 1)
        p_prev2 = p_prev1
        p_prev1 = p_n
    return p_n


def PDD(x, n, y):
    dim = x.shape[0]
    N = x.shape[1]
    phi = [np.ones(N)]
    mapping_list = [[0] * dim]

    for i in range(dim):
        for j in range(1, n + 1):
            phi.append(basis(x[i, :], j))
            mapping = [0] * dim
            mapping[i] = 1
            mapping_list.append(mapping)

    if y >= 2:
        for i in range(2, n + 1):
            for j in range(1, i):
                for k in range(dim):
                    for l in range(k + 1, dim):
                        phi.append(basis(x[k, :], j) * basis(x[l, :], i - j))
                        mapping = [0] * dim
                        mapping[k] = 1
                        mapping[l] = 1
                        mapping_list.append(mapping)

    return np.array(phi).T, np.array(mapping_list).T


def find_optimal_degree(input_data, output_data, max_n=15, max_y=2,
                         val_ratio=0.2, patience=3, tol=1e-4, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    N_samples = input_data.shape[1]
    perm = rng.permutation(N_samples)
    n_val = max(int(N_samples * val_ratio), 1)
    val_idx, train_idx = perm[:n_val], perm[n_val:]

    X_train, X_val = input_data[:, train_idx], input_data[:, val_idx]
    Y_train, Y_val = output_data[train_idx], output_data[val_idx]

    best_n, best_y = 1, 1
    best_val_r2 = -float("inf")
    no_improve = 0

    for n in range(1, max_n + 1):
        for y in range(1, max_y + 1):
            exp_train, mapping = PDD(X_train, n, y)
            k = exp_train.shape[1]
            if k >= X_train.shape[1] - 1:
                continue

            Ci = np.linalg.pinv(exp_train) @ Y_train
            exp_val, _ = PDD(X_val, n, y)
            predicted_val = exp_val @ Ci

            ss_res = np.sum((Y_val - predicted_val) ** 2)
            ss_tot = np.sum((Y_val - np.mean(Y_val)) ** 2)
            val_r2 = 1 - (ss_res / ss_tot)

            if val_r2 > best_val_r2 + tol:
                best_val_r2 = val_r2
                best_n, best_y = n, y
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                if verbose:
                    print(f"선택된 조합: n={best_n}, y={best_y} (검증 R^2: {best_val_r2:.6f})")
                return best_n, best_y

    if verbose:
        print(f"선택된 조합: n={best_n}, y={best_y} (검증 R^2: {best_val_r2:.6f})")
    return best_n, best_y

## 벤치마크 함수 정의

`min(branch_a, branch_b)`으로 꺾임(kink)을 만들되, 두 branch가 **`x1`과 `x2` 둘 다에 의존**하게 해서 진짜 2변수 상호작용이 생기게 함(순수 `x1`만의 함수면 PDD의 1변수 성분만으로 풀리는 쉬운 문제가 돼서 공정한 비교가 안 됨). `x3`은 기여가 거의 없는 변수(민감도 스크리닝 대상 흉내).

In [ ]:
def kink_function(x1, x2, x3):
    branch_a = 1.2 * np.sin(2 * x1) + 0.6 * x2
    branch_b = 1.2 * np.cos(2 * x1) - 0.6 * x2
    return np.minimum(branch_a, branch_b) + 0.05 * x3


domain_min = np.array([[-2.0], [-2.0], [-2.0]])
domain_max = np.array([[2.0], [2.0], [2.0]])

n_train = 150
n_test = 1000

rng = np.random.default_rng(rng_seed)
X_train_phys = rng.uniform(-2, 2, size=(3, n_train))
X_test_phys = rng.uniform(-2, 2, size=(3, n_test))

Y_train = kink_function(*X_train_phys)
Y_test = kink_function(*X_test_phys)

X_train_scaled = theoretical_scale(X_train_phys, domain_min, domain_max)
X_test_scaled = theoretical_scale(X_test_phys, domain_min, domain_max)

print(f"학습 표본 {n_train}개, 검증 표본 {n_test}개")

## PDD 적합 및 검증

In [ ]:
opt_n, opt_y = find_optimal_degree(X_train_scaled, Y_train)

exp_train, mapping = PDD(X_train_scaled, opt_n, opt_y)
Ci = np.linalg.pinv(exp_train) @ Y_train

exp_test, _ = PDD(X_test_scaled, opt_n, opt_y)
Y_pred_pdd = exp_test @ Ci

ss_res = np.sum((Y_test - Y_pred_pdd) ** 2)
ss_tot = np.sum((Y_test - np.mean(Y_test)) ** 2)
r2_pdd = 1 - ss_res / ss_tot
rmse_pdd = np.sqrt(np.mean((Y_test - Y_pred_pdd) ** 2))

print(f"[PDD] 검증 R^2={r2_pdd:.4f}, RMSE={rmse_pdd:.4f}")

## 크리깅 적합 및 검증

Matern 커널의 스무스니스 파라미터(`nu`)를 임의로 고정하지 않고, **학습 데이터만으로 교차검증**해서 고름(검증/테스트 데이터는 안 씀 — PDD의 차수 선택과 동일한 원칙).

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

best_nu, best_cv_score = None, -np.inf
for nu in [0.5, 1.5, 2.5]:
    kernel_cv = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=nu) + WhiteKernel(1e-4)
    gpr_cv = GaussianProcessRegressor(kernel=kernel_cv, normalize_y=True, n_restarts_optimizer=3, random_state=0)
    scores = cross_val_score(gpr_cv, X_train_scaled.T, Y_train, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")
    print(f"nu={nu}: 학습셋 5-fold CV R^2={scores.mean():.4f}")
    if scores.mean() > best_cv_score:
        best_cv_score = scores.mean()
        best_nu = nu

print(f"선택된 nu={best_nu}")

kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=best_nu) + WhiteKernel(1e-4)
gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5, random_state=0)
gpr.fit(X_train_scaled.T, Y_train)

Y_pred_gpr = gpr.predict(X_test_scaled.T)

ss_res = np.sum((Y_test - Y_pred_gpr) ** 2)
ss_tot = np.sum((Y_test - np.mean(Y_test)) ** 2)
r2_gpr = 1 - ss_res / ss_tot
rmse_gpr = np.sqrt(np.mean((Y_test - Y_pred_gpr) ** 2))

print(f"[크리깅] 검증 R^2={r2_gpr:.4f}, RMSE={rmse_gpr:.4f}")

## 비교

In [ ]:
print("=" * 40)
print(f"{'모델':<10} | {'R^2':>10} | {'RMSE':>10}")
print("-" * 40)
print(f"{'PDD':<10} | {r2_pdd:>10.4f} | {rmse_pdd:>10.4f}")
print(f"{'크리깅':<10} | {r2_gpr:>10.4f} | {rmse_gpr:>10.4f}")
print("=" * 40)

# 1D 단면: x2=x3=0으로 고정, x1만 바꿔가며 실제 함수 vs PDD vs 크리깅 비교
# (matplotlib 기본 폰트에 한글이 없어 그래프 텍스트는 영문으로 표기)
x1_line = np.linspace(-2, 2, 300)
X_line_phys = np.vstack([x1_line, np.zeros_like(x1_line), np.zeros_like(x1_line)])
X_line_scaled = theoretical_scale(X_line_phys, domain_min, domain_max)

Y_line_true = kink_function(*X_line_phys)
exp_line, _ = PDD(X_line_scaled, opt_n, opt_y)
Y_line_pdd = exp_line @ Ci
Y_line_gpr = gpr.predict(X_line_scaled.T)

plt.figure(figsize=(8, 5))
plt.plot(x1_line, Y_line_true, 'k-', linewidth=2, label='true function (kinked)')
plt.plot(x1_line, Y_line_pdd, 'r--', linewidth=1.5, label=f'PDD (R^2={r2_pdd:.3f})')
plt.plot(x1_line, Y_line_gpr, 'b--', linewidth=1.5, label=f'Kriging (R^2={r2_gpr:.3f})')
plt.xlabel('x1')
plt.ylabel('f(x1, 0, 0)')
plt.title('PDD vs Kriging on a kinked response surface')
plt.legend()
plt.tight_layout()
plt.savefig('pdd_vs_kriging_kink_slice.png', dpi=120)
plt.show()